# **Session 3 : LangChain**

In [ ]:
!pip -q install -U langchain langchain-Groq langsmith

In [ ]:
import os
import getpass
from google.colab import userdata
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = "<put_your_api_key>"
os.environ["GROQ_API_KEY"] = getpass.getpass("Groq API Key:")
#os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')

In [ ]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate, SystemMessagePromptTemplate, HumanMessagePromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [ ]:
#question = "how many Egyptian football team won African Cup?"
question = "كم مرة فاز المنتخب المصري لكرة القدم ببطولة الامم الافريقية؟"

In [ ]:
lang_model = ChatGroq(model="moonshotai/kimi-k2-instruct")
lang_template = ChatPromptTemplate.from_messages(
    [
        SystemMessagePromptTemplate.from_template("You are a linguistic analyst. your job is to read user question and reply ONLY with langue ISO language Code, for example: ar for arabic, en for english."),
        HumanMessagePromptTemplate.from_template("user question: {question}")
        ]
    )
Parser = StrOutputParser()

lang_chain = lang_template | lang_model | Parser
lang_response = lang_chain.invoke(question)
print(lang_response)

In [ ]:
en_model = ChatGroq(model="moonshotai/kimi-k2-instruct")
en_template = ChatPromptTemplate.from_messages(
    [
        SystemMessagePromptTemplate.from_template("You are ENGLISH language AI Assistant answer question in English language ONLY."),
        HumanMessagePromptTemplate.from_template("user question: {question}")
        ]
    )
Parser = StrOutputParser()

en_chain = en_template | en_model | Parser

In [ ]:
ar_model = ChatGroq(model="gemma2-9b-it")
ar_template = ChatPromptTemplate.from_messages(
    [
        SystemMessagePromptTemplate.from_template("You are ARABIC language AI Assistant answer question in Arabic language ONLY."),
        HumanMessagePromptTemplate.from_template("user question: {question}")
        ]
    )
Parser = StrOutputParser()

ar_chain = ar_template | ar_model | Parser

In [ ]:
if lang_response == "ar":
  ar_response = ar_chain.invoke(question)
  print(lang_response)
  print(ar_response)
elif lang_response == "en":
  en_response = en_chain.invoke(question)
  print(lang_response)
  print(en_response)

##1) LangSmith

In [ ]:
groq_model = ChatGroq(model="moonshotai/kimi-k2-instruct")

In [ ]:
prompt_template = ChatPromptTemplate.from_messages(
    [
        SystemMessagePromptTemplate.from_template("You are an Egyptian TV football news analyst assistant. You answer the user’s questions with at least two memorable match moments."),
        HumanMessagePromptTemplate.from_template("how many {team_nationality} football team won {continental} Cup?")
        ]
    )
Parser = StrOutputParser()
chain = prompt_template | groq_model | Parser
respond = chain.invoke({"team_nationality": "Egyptian", "continental": "African"})
print(respond)

##2) Coercion Functions

In [ ]:
# Example Two functions
#1)
def nationality (country: str):
    if country == "Egypt":
        return "Egyptian"
    elif country == "Italy":
        return "Italian"
    else:
        return "Unknown"
#2)
def cup_nation (country: str):
    if country == "Egypt":
        return "African"
    elif country == "Italy":
        return "Euro"
    else:
        return "Unknown"

In [ ]:
chain = (
    {
        "team_nationality": lambda x1: nationality(x1),
        "continental": lambda x2: cup_nation(x2)
     }                                                  # <==== Coercion Functions
    | prompt_template | groq_model | Parser
)

chain

In [ ]:
chain_respond = chain.invoke("Egypt")
print(chain_respond)

##3) Runnable Parallels Chains

In [ ]:
poem_template = ChatPromptTemplate.from_messages(
    [
        SystemMessagePromptTemplate.from_template(
            """
You are a skilled Arabic poet in a poetry competition.
When given a country, write a titled poem in classical Arabic that follows poetic meter and rhyme.
The poem should highlight the country’s heritage, nature, and modern culture, inspiring admiration and tourism.
Respond only in Arabic and DO NOT add translate your response to English.
            """
            ),
        HumanMessagePromptTemplate.from_template("Write a poem about country: {country}")
        ]
    )

In [ ]:
model_meta = ChatGroq(model="meta-llama/llama-4-maverick-17b-128e-instruct")
model_gemma = ChatGroq(model="gemma2-9b-it")

In [ ]:
chain_meta = poem_template | model_meta | Parser    # Poem 1

In [ ]:
chain_gemma = poem_template | model_gemma | Parser  # Poem 2

In [ ]:
eval_template = ChatPromptTemplate.from_messages(
    [
        SystemMessagePromptTemplate.from_template(
            "You are a jury member in an Arabic poetry competition. Compare the poems decide and announce the winner name who wrote the best poem for promoting tourism. Respond only in Arabic."
        ),
        HumanMessagePromptTemplate.from_template(
            """
            Meta Poem :
            {pom1}

            Gemma Poem :
            {pom2}
            """
            )
        ]
    )

In [ ]:
eval_model = ChatGroq(model="openai/gpt-oss-120b")

In [ ]:
from langchain_core.runnables import RunnableParallel

In [ ]:
eval_chain = RunnableParallel(pom1=chain_meta, pom2=chain_gemma) | eval_template | eval_model | Parser

In [ ]:
# Invoke the composed chain
evaluation = eval_chain.invoke({"country": "Egypt"})
print(evaluation)

**التحكيم**

**المقارنة بين القصيدتين**

| العنصر | قصيدة «Meta Poem» | قصيدة «Gemma Poem» |
|--------|-------------------|--------------------|
| **المحتوى الترويجي** | تركز على إبراز مصر كقلب التاريخ والحضارة، وتدعو القارئ إلى زيارة “البلاد التي تجمع بين القديم والحديث”. توضح بوضوح وجود المعالم السياحية (الأهرام، النيل) وتصفها بعبارات جذابة مثل “لؤلؤة نقية” و“روضة غنّاءة”. | تذكر المعالم الأساسية (الأهرام، النيل، الصحراء) وتضيف إشارة إلى مصر الحديثة وعلمائها، لكن الدعوة للزيارة ليست واضحة بقدر ما هي إشادة عامة بالمكان. |
| **اللغة والأسلوب** | يستخدم أسلوباً شعرياً فصيحاً، توالي الأبيات المتقن، واستخدام تراكيب مثل “تبدو كأنها لؤلؤة نقية” و“تجمع بين القديم والحديث” يضيف رونقاً إعلانيًا. | الأسلوب أبسط وأقل تجانساً؛ توجد خلطات بين الفصحى والعامية (مثل “سphinx”) وتكرار بعض الكلمات دون تناغم إيقاعي. |
| **الإيقاع والوزن** | يتسم بإيقاع ثابت نسبياً، مع توازن بين الجمل الطويلة والقصيرة، ما يسهل قراءته ويساهم في تأثيره الدعائي. | الإيقاع غير منتظم، وتقطيع الأبيات غير متساوٍ، ما يضعف من قوة 

##4) RunnablePassthrough

In [ ]:
from langchain_core.runnables import RunnablePassthrough

eval_chain = (
    RunnableParallel(pom1=chain_meta, pom2=chain_gemma)
    | RunnablePassthrough.assign(
        evlauatn=eval_template | eval_model | Parser
        )
    |
         (lambda inputs: f"Meta Poem:\n{inputs['pom1']}\n\n"
                      f"Gemma Poem:\n{inputs['pom2']}\n\n"
                      f"Evalution:\n{inputs['evlauatn']}")
)

In [ ]:
result = eval_chain.invoke({"country": "Egypt"})
print(result)

Meta Poem:
"مصر الأهرام"

بِأَرضِكَ يا مِصرُ تَزهُرُ الحَياةُ
وَتَعلو الهِمَمُ وَتَسطَعُ النَجاةُ
أَهراماتُكَ شامِخاتٌ كَالجِبالِ
وَنيلُكَ يَجري كَالدُجى وَالرِمالِ
تاريخُكَ مَجيدٌ وَحَضارَةٌ خالِدَةٌ
وَشعبُكَ كَريمٌ وَطَبيعةٌ ساحِرَةٌ
مِن سيناءَ إِلى الإِسكندرِيَّةِ
تَجري الدُنيا بِخَيرِها وَالبَرَكَةِ
في أَرضِكَ يا مِصرُ تَجتَمِعُ الأَماني
وَتَتَجَلّى رَوائعُ الفَنِّ وَالعِلمِ
زُرْ مِصرَ وَتَأَمَّلْ عَظَمَتَها
وَاسْتَنشِقْ نَسيمَ النيلِ وَالرِمالِ
فَمِصرُ أَرضُ الحَضاراتِ وَالتاريخِ
وَمَصرُ أَرضُ العِزِّ وَالاِزدهارِ.

Gemma Poem:
##  فراعنة الأمس وتايمس اليوم

بنيانِ قدماءٍ على ضفافٍ نيل،
جبالٌ شامخةٌ في سماءٍ كئيبة.
فراعنةٌ عظماءٌ، من ذهبٍ وسحر،
أبصارُهم تنظر، عبرَ الزمنِ، إلى الأبدِ.

وَقصرُهُم لا يزال، يروي قصصَ،
من تاريخٍ عريقٍ، بِالحجارةِ، تحتِ الأرصَفِ.
الرمالُ تنسابُ، على شواطئ البحرِ،
مُنَظّرةٌ، من كُلِ البُعد، شمسًَ، في غروبِها.

مصرُ الحديثةُ، شامخةٌ، مبهرةٌ،
مُبانيٌّ عاليةٌ، في سماءٍ مفتوحة.
أُناسٌ مبدعون، في الفنِ، والعلومِ،
يُرسّونَ شغفَ، في كلِ جِدْةٍ، وِمُحِبَةٍ.

فأع